In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import pickle

In [2]:
df = pd.read_csv("meddata.csv")

df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [3]:
print(df.info())

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDataset Shape:")
print(df.shape)

<class 'pandas.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   str    
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   str    
 5   region    1338 non-null   str    
 6   charges   1338 non-null   float64
dtypes: float64(2), int64(2), str(3)
memory usage: 94.5 KB
None

Missing Values:
age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64

Dataset Shape:
(1338, 7)


In [4]:
X = df.drop("charges", axis=1)
y = df["charges"]

In [5]:
categorical_features = ["sex", "smoker", "region"]
numerical_features = ["age", "bmi", "children"]

In [6]:
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ],
    remainder="passthrough"
)

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [8]:
model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=200,
        random_state=42
    ))
])

In [9]:
model.fit(X_train, y_train)

print("Model trained successfully!")

Model trained successfully!


In [10]:
predictions = model.predict(X_test)

print("R2 Score :", r2_score(y_test, predictions))
print("MAE      :", mean_absolute_error(y_test, predictions))
print("RMSE     :", np.sqrt(mean_squared_error(y_test, predictions)))

R2 Score : 0.8634585240286198
MAE      : 2528.020213412112
RMSE     : 4604.116738373523


In [11]:
with open("insurance_model.pkl", "wb") as file:
    pickle.dump(model, file)

print("Model saved as insurance_model.pkl")

Model saved as insurance_model.pkl


In [12]:
with open("insurance_model.pkl", "rb") as file:
    loaded_model = pickle.load(file)

sample = pd.DataFrame({
    "age": [25],
    "sex": ["male"],
    "bmi": [28.5],
    "children": [0],
    "smoker": ["no"],
    "region": ["southwest"]
})

prediction = loaded_model.predict(sample)

print("Predicted Insurance Charge: $", round(prediction[0], 2))

Predicted Insurance Charge: $ 2645.27
